# Klasifikasi DemogPairs Menggunakan ViT (Wajah dan Umur) & Random Forest

In [1]:
import numpy as np
import utils as u
import joblib
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, ParameterGrid
from imblearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
from tqdm import tqdm

joblib.parallel_backend('threading')

## Load Dataset

In [2]:
data = u.load_demogpairs()
pd.DataFrame(data)

,db_code,image_path,full_path,label,label_idx
0,CWF,able_wanamakok/002.jpg,dataset/demogpairs/images\able_wanamakok/002.jpg,Asian_Females,5
1,CWF,able_wanamakok/004.jpg,dataset/demogpairs/images\able_wanamakok/004.jpg,Asian_Females,5
2,CWF,able_wanamakok/007.jpg,dataset/demogpairs/images\able_wanamakok/007.jpg,Asian_Females,5
3,CWF,able_wanamakok/008.jpg,dataset/demogpairs/images\able_wanamakok/008.jpg,Asian_Females,5
4,CWF,able_wanamakok/012.jpg,dataset/demogpairs/images\able_wanamakok/012.jpg,Asian_Females,5
...,...,...,...,...,...
10795,CWF,zachary_quinto/177.jpg,dataset/demogpairs/images\zachary_quinto/177.jpg,White_Males,3
10796,CWF,zachary_quinto/214.jpg,dataset/demogpairs/images\zachary_quinto/214.jpg,White_Males,3
10797,CWF,zachary_quinto/217.jpg,dataset/demogpairs/images\zachary_quinto/217.jpg,White_Males,3
10798,CWF,zachary_quinto/218.jpg,dataset/demogpairs/images\zachary_quinto/218.jpg,White_Males,3


## Load Fitur

In [3]:
face_features = joblib.load('features/demogpairs_vit-face.pkl')
age_features = joblib.load('features/demogpairs_vit-age.pkl')
features = {}
for d in tqdm(data):
    key = d['image_path']
    features[key] = np.array(list(face_features[key]) + list(age_features[key]))
print('Jumlah fitur per gambar:', np.array(features[list(features.keys())[0]]).shape[0])

100%|█████████████████████████████████████████████████████████████████████████| 10800/10800 [00:00<00:00, 10877.65it/s]

Jumlah fitur per gambar: 1536


## Split Data

In [4]:
X = np.array([features[d['image_path']] for d in data])
y = np.array([d['label_idx'] for d in data])
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
print((len(X_train), len(X_test)))

(8640, 2160)


## Kombinasi Parameter

In [5]:
grid_params = [
    {
        'scaler': [None, MinMaxScaler()],
        'pca': [None, PCA(n_components=0.5), PCA(n_components=0.75)],
        
        'classifier': [RandomForestClassifier(random_state=42)],
        'classifier__n_estimators': [100, 200],
        'classifier__max_depth': [None, 20, 30],
        'classifier__min_samples_split': [2, 5],
        'classifier__min_samples_leaf': [1, 2],
        'classifier__max_features': ['sqrt', 'log2'],
    },
]

pipeline = Pipeline(steps=[
    ('scaler', None),
    ('pca', None),
    ('classifier', None)
])

skv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scoring = {
    'accuracy': 'accuracy', 
    'f1': 'f1_macro', 
    'precision': 'precision_macro', 
    'recall': 'recall_macro',
}

grid_models = {}
for params in grid_params:
    key = str(params['classifier'][0]).split('(')[0]
    grid_models[key] = GridSearchCV(
        estimator=pipeline,
        param_grid=params,
        cv=skv, refit='accuracy',
        scoring=scoring, n_jobs=int(joblib.cpu_count() * 0.6),
        verbose=1, error_score='raise',
        return_train_score=True
    )
    print(f'{key}: {len(ParameterGrid(params))} kombinasi')

RandomForestClassifier: 288 kombinasi


## Klasifikasi

In [6]:
evaluation_results, fold_results = u.evaluate_models(
    grid_models, 
    X_train, y_train,
    X_test, y_test,
    target_names=u.demogpairs_classes,
    model_prefix="models/clf_demogpairs_rf_vit-face-age_",
    results_path="results/demogpairs_rf_vit-face-age_"
)
sorted_results = pd.DataFrame(evaluation_results).sort_values(by="test_accuracy", ascending=False).to_dict("records")
u.html_br()
_dtable = u.display_table(sorted_results)

Evaluating: RandomForestClassifier


{'classifier': 'RandomForestClassifier', 'classifier__max_depth': None, 'classifier__max_features': 'sqrt', 'classifier__min_samples_leaf': 1, 'classifier__min_samples_split': 2, 'classifier__n_estimators': 200, 'pca': 'PCA', 'scaler': None}


Accuracy  : 0.8578703703703704
Precision : 0.85780459464061
Recall    : 0.8578703703703705
F1 Score  : 0.857295189036103
               precision    recall  f1-score   support

Asian_Females     0.8633    0.8944    0.8786       360
  Asian_Males     0.8861    0.8861    0.8861       360
Black_Females     0.8194    0.8444    0.8317       360
  Black_Males     0.8859    0.9278    0.9064       360
White_Females     0.8611    0.7750    0.8158       360
  White_Males     0.8310    0.8194    0.8252       360

     accuracy                         0.8579      2160
    macro avg     0.8578    0.8579    0.8573      2160
 weighted avg     0.8578    0.8579    0.8573      2160



Class,OvR Accuracy,Precision,Recall,F1-Score,Support
Asian_Females,0.9587962962962963,0.8632707774798928,0.8944444444444445,0.8785811732605731,360
Asian_Males,0.962037037037037,0.8861111111111111,0.8861111111111111,0.8861111111111111,360
Black_Females,0.9430555555555555,0.8194070080862533,0.8444444444444444,0.8317373461012311,360
Black_Males,0.9680555555555556,0.8859416445623343,0.9277777777777778,0.9063772048846677,360
White_Females,0.9416666666666667,0.8611111111111112,0.775,0.8157894736842106,360
White_Males,0.9421296296296297,0.8309859154929577,0.8194444444444444,0.8251748251748252,360


Confusion matrix saved: images\cm_rf_vit-face-age_RandomForestClassifier.png



Confusion Matrix:
                         Asian_Females       Asian_Males     Black_Females       Black_Males     White_Females       White_Males
       Asian_Females               322                 0                13                16                 9                 0
         Asian_Males                 2               319                 4                 2                12                21
       Black_Females                 9                 0               304                21                 3                23
         Black_Males                 7                 5                12               334                 0                 2
       White_Females                32                21                12                 2               279                14
         White_Males                 1                15                26                 2                21               295


model_name,model_file_path,best_parameters,test_accuracy,test_f1,test_precision,test_recall,parameter_combinations
RandomForestClassifier,models/clf_demogpairs_rf_vit-face-age_RandomForestClassifier.pkl,"{'classifier': 'RandomForestClassifier', 'classifier__max_depth': None, 'classifier__max_features': 'sqrt', 'classifier__min_samples_leaf': 1, 'classifier__min_samples_split': 2, 'classifier__n_estimators': 200, 'pca': 'PCA', 'scaler': None}",0.8578703703703704,0.857295189036103,0.85780459464061,0.8578703703703705,288


In [7]:
model, training_time = u.load_object('models/clf_demogpairs_rf_vit-face-age_RandomForestClassifier.pkl')
u.h(5, 'Waktu Pelatihan (Jobs)')
u.seconds_to_time(round(training_time))

{'input_seconds': 6311.0,
 'days': 0,
 'hours': 1,
 'minutes': 45,
 'seconds': 11.0,
 'text': '0 hari 1 jam 45 menit 11.0 detik'}

In [8]:
u.h(5, 'Waktu Pelatihan')
times = [fr['Train Time Mean'] * 5 for fr in fold_results]
u.seconds_to_time(round(np.sum(times) + model.refit_time_))

{'input_seconds': 24456.0,
 'days': 0,
 'hours': 6,
 'minutes': 47,
 'seconds': 36.0,
 'text': '0 hari 6 jam 47 menit 36.0 detik'}

In [9]:
_dtable = u.display_table(fold_results, n_items=[4, 4], column_widths=['5%', '45%', '5%', '5%', '5%', '5%', '5%', '5%', '5%', '5%', '5%', '5%'])

No,Params,Fold 1,Fold 2,Fold 3,Fold 4,Fold 5,Accuracy Mean,F1 Score Mean,Precision Mean,Recall Mean,Train Time Mean
1,"{'classifier': 'RandomForestClassifier', 'classifier__max_depth': None, 'classifier__max_features': 'sqrt', 'classifier__min_samples_leaf': 1, 'classifier__min_samples_split': 2, 'classifier__n_estimators': 200, 'pca': 'PCA', 'scaler': None}",0.886,0.8709,0.8646,0.8721,0.8663,0.872,0.8715,0.8721,0.872,18.8863
2,"{'classifier': 'RandomForestClassifier', 'classifier__max_depth': 20, 'classifier__max_features': 'sqrt', 'classifier__min_samples_leaf': 1, 'classifier__min_samples_split': 5, 'classifier__n_estimators': 200, 'pca': 'PCA', 'scaler': None}",0.8843,0.8686,0.8663,0.8663,0.8709,0.8713,0.8709,0.8712,0.8713,18.0591
3,"{'classifier': 'RandomForestClassifier', 'classifier__max_depth': 20, 'classifier__max_features': 'log2', 'classifier__min_samples_leaf': 2, 'classifier__min_samples_split': 2, 'classifier__n_estimators': 200, 'pca': 'PCA', 'scaler': None}",0.8843,0.8686,0.8634,0.875,0.8628,0.8708,0.8704,0.871,0.8708,16.946
4,"{'classifier': 'RandomForestClassifier', 'classifier__max_depth': 30, 'classifier__max_features': 'sqrt', 'classifier__min_samples_leaf': 1, 'classifier__min_samples_split': 2, 'classifier__n_estimators': 200, 'pca': 'PCA', 'scaler': None}",0.8854,0.8669,0.8634,0.8721,0.8657,0.8707,0.8703,0.8708,0.8707,19.0386
...,...,...,...,...,...,...,...,...,...,...,...
285,"{'classifier': 'RandomForestClassifier', 'classifier__max_depth': 30, 'classifier__max_features': 'log2', 'classifier__min_samples_leaf': 2, 'classifier__min_samples_split': 2, 'classifier__n_estimators': 100, 'pca': 'PCA', 'scaler': 'MinMaxScaler'}",0.8762,0.8553,0.8524,0.8605,0.8495,0.8588,0.8584,0.8589,0.8588,9.7462
286,"{'classifier': 'RandomForestClassifier', 'classifier__max_depth': None, 'classifier__max_features': 'log2', 'classifier__min_samples_leaf': 2, 'classifier__min_samples_split': 2, 'classifier__n_estimators': 100, 'pca': 'PCA', 'scaler': 'MinMaxScaler'}",0.875,0.8547,0.8524,0.8605,0.8495,0.8584,0.858,0.8585,0.8584,9.7151
287,"{'classifier': 'RandomForestClassifier', 'classifier__max_depth': 20, 'classifier__max_features': 'log2', 'classifier__min_samples_leaf': 2, 'classifier__min_samples_split': 2, 'classifier__n_estimators': 100, 'pca': 'PCA', 'scaler': 'MinMaxScaler'}",0.8727,0.8565,0.8559,0.8559,0.8478,0.8578,0.8573,0.858,0.8578,10.3506
288,"{'classifier': 'RandomForestClassifier', 'classifier__max_depth': 20, 'classifier__max_features': 'log2', 'classifier__min_samples_leaf': 1, 'classifier__min_samples_split': 5, 'classifier__n_estimators': 100, 'pca': 'PCA', 'scaler': 'MinMaxScaler'}",0.8709,0.8576,0.8507,0.8542,0.8542,0.8575,0.8571,0.8576,0.8575,9.9864
